# 2-Tier SAR Ship Classifier — Part 1
## Setup + Tier 1 Training (Binary: Cargo vs Not-Cargo)

All three architectures trained as binary classifiers:
- **resnet** — ResNet-18 transfer
- **bam** — ResNet-18 + CBAM attention
- **swin** — Swin Transformer

Each trains 4 models: os1, os2, fusar, joint.
Weights and splits are saved so Part 2 can load them directly.

## Configuration

In [ ]:
import gc
import json
import os
import sys
import zipfile
from datetime import datetime
from pathlib import Path
from typing import Dict, List, Optional, Tuple

import gdown
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from PIL import Image
from sklearn.metrics import classification_report
from sklearn.model_selection import train_test_split
from torch.optim import Adam
from torch.optim.lr_scheduler import CosineAnnealingLR
from torch.utils.data import DataLoader, Dataset, Subset
from torchvision import transforms

RANDOM_SEED = 42
torch.manual_seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

EPOCHS     = 15
BATCH_SIZE = 32
LR         = 1e-4

PROJECT_ROOT     = Path(".").resolve().parent
DATA_DIR         = PROJECT_ROOT / "data" / "classification"
RUN_TS           = datetime.now().strftime("%Y%m%d_%H%M%S")
OUT_DIR          = Path(".").resolve() / "outputs" / f"2tier_{RUN_TS}"
WEIGHTS_DIR      = OUT_DIR / "weights"
MISC_DIR         = OUT_DIR / "misc"
DRIVE_FOLDER_URL = "https://drive.google.com/drive/folders/19jLMSzHChVLk-vVAg2muNN2OALzksWob"

for _d in (WEIGHTS_DIR, MISC_DIR):
    _d.mkdir(parents=True, exist_ok=True)

os.environ["CUDA_LAUNCH_BLOCKING"] = "1"
os.environ["TORCH_USE_CUDA_DSA"]   = "1"
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"Device : {DEVICE}")
if DEVICE.type == "cuda":
    print(f"GPU    : {torch.cuda.get_device_name(0)}")
print(f"Output : {OUT_DIR}")

## Model Imports

In [ ]:
_SRC = Path(".").resolve()
if str(_SRC) not in sys.path:
    sys.path.insert(0, str(_SRC))

from models.resnet_transfer import CNN_resnet_transfer
from models.resnet_attention import resnet_attention
from models.swin_transfer    import swin_transfer

def get_model(arch: str, num_classes: int) -> nn.Module:
    if arch == "resnet": return CNN_resnet_transfer(num_classes)
    if arch == "bam":    return resnet_attention(num_classes)
    if arch == "swin":   return swin_transfer(num_classes)
    raise ValueError(f"Unknown architecture: {arch}")

print("Loaded: CNN_resnet_transfer, resnet_attention, swin_transfer")

## Load Label CSVs

In [ ]:
DATA_DIR.mkdir(parents=True, exist_ok=True)

if not (DATA_DIR / "opensar1_labels.csv").exists():
    print("Downloading label CSVs from Google Drive...")
    gdown.download_folder(DRIVE_FOLDER_URL, output=str(DATA_DIR), quiet=False, use_cookies=False)
else:
    print(f"Data already present at {DATA_DIR}")

opensar1 = pd.read_csv(DATA_DIR / "opensar1_labels.csv")
opensar2 = pd.read_csv(DATA_DIR / "opensar2_labels.csv")
fusar    = pd.read_csv(DATA_DIR / "fusar_labels.csv")

for df in (opensar1, opensar2, fusar):
    df["path"] = df["path"].apply(lambda p: str(PROJECT_ROOT / p))
    df.drop(df[df["label"] == "unknown"].index, inplace=True)
    df.reset_index(drop=True, inplace=True)

print(f"os1: {len(opensar1)}  os2: {len(opensar2)}  fusar: {len(fusar)}")
fusar.head()

## Label Setup

In [ ]:
joint = pd.concat([opensar1, opensar2, fusar], ignore_index=True)

# Tier 1: binary
for df in (opensar1, opensar2, fusar, joint):
    df["t1_label_id"] = df["is_cargo"].astype(int)

TIER1_CLASS_NAMES = ["not_cargo", "cargo"]
TIER1_NUM_CLASSES = 2

# Tier 2: multi-class non-cargo (labels defined here, used in Part 2)
os1_nc   = opensar1[opensar1["is_cargo"] == 0].reset_index(drop=True)
os2_nc   = opensar2[opensar2["is_cargo"] == 0].reset_index(drop=True)
fusar_nc = fusar[fusar["is_cargo"] == 0].reset_index(drop=True)

nc_labels = sorted(
    set(os1_nc["label"]).union(set(os2_nc["label"])).union(set(fusar_nc["label"]))
)
TIER2_LABEL_TO_INT = {label: i for i, label in enumerate(nc_labels)}
TIER2_CLASS_NAMES  = nc_labels
TIER2_NUM_CLASSES  = len(nc_labels)

for df in (os1_nc, os2_nc, fusar_nc):
    df["t2_label_id"] = df["label"].map(TIER2_LABEL_TO_INT)

print("Tier 1 label distribution:")
for name, df in [("os1", opensar1), ("os2", opensar2), ("fusar", fusar), ("joint", joint)]:
    c = df["t1_label_id"].value_counts().sort_index()
    print(f"  {name}: not_cargo={c.get(0,0)}  cargo={c.get(1,0)}")

print(f"\nTier 2 mapping: {TIER2_LABEL_TO_INT}")

## Dataset Class and Transforms

In [ ]:
_MEAN = [0.485, 0.456, 0.406]
_STD  = [0.229, 0.224, 0.225]

transform_train = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.Grayscale(num_output_channels=3),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.RandomRotation(15, fill=0),
    transforms.ToTensor(),
    transforms.Normalize(mean=_MEAN, std=_STD),
])
transform_eval = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.Grayscale(num_output_channels=3),
    transforms.ToTensor(),
    transforms.Normalize(mean=_MEAN, std=_STD),
])

class SARDataset(Dataset):
    """label_col: 't1_label_id' for Tier 1, 't2_label_id' for Tier 2."""
    def __init__(self, dataframe: pd.DataFrame, label_col: str, transform=None):
        self.df        = dataframe.reset_index(drop=True)
        self.label_col = label_col
        self.transform = transform

    def __len__(self) -> int:
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        try:
            image = Image.open(row["path"]).convert("L")
        except (FileNotFoundError, OSError) as e:
            raise RuntimeError(f"Could not load image at index {idx}: {row['path']}") from e
        if self.transform:
            image = self.transform(image)
        return image, torch.tensor(int(row[self.label_col]), dtype=torch.long)

print("SARDataset and transforms defined.")

## Build Splits and DataLoaders

In [ ]:
_PIN = DEVICE.type == "cuda"
_NW  = 2

def build_splits(df: pd.DataFrame, label_col: str) -> dict:
    """Stratified 72/8/20 train/val/test split with DataLoaders."""
    labels = df[label_col].values
    idx    = np.arange(len(df))

    trainval_idx, test_idx = train_test_split(
        idx, test_size=0.20, stratify=labels, random_state=RANDOM_SEED
    )
    train_idx, val_idx = train_test_split(
        trainval_idx, test_size=0.10,
        stratify=labels[trainval_idx], random_state=RANDOM_SEED
    )

    ds_train = SARDataset(df, label_col, transform=transform_train)
    ds_eval  = SARDataset(df, label_col, transform=transform_eval)

    return {
        "train_idx"    : list(train_idx),
        "val_idx"      : list(val_idx),
        "test_idx"     : list(test_idx),
        "train_loader" : DataLoader(Subset(ds_train, train_idx), batch_size=BATCH_SIZE,
                                    shuffle=True,  num_workers=_NW, pin_memory=_PIN,
                                    persistent_workers=bool(_NW)),
        "val_loader"   : DataLoader(Subset(ds_eval,  val_idx),   batch_size=BATCH_SIZE,
                                    shuffle=False, num_workers=_NW, pin_memory=_PIN,
                                    persistent_workers=bool(_NW)),
        "test_loader"  : DataLoader(Subset(ds_eval,  test_idx),  batch_size=BATCH_SIZE,
                                    shuffle=False, num_workers=_NW, pin_memory=_PIN,
                                    persistent_workers=bool(_NW)),
    }

# Tier 1 splits
print("Building Tier 1 splits...")
tier1_domains = {
    "os1"  : build_splits(opensar1, "t1_label_id"),
    "os2"  : build_splits(opensar2, "t1_label_id"),
    "fusar": build_splits(fusar,    "t1_label_id"),
    "joint": build_splits(joint,    "t1_label_id"),
}

# Tier 2 splits (also built here so indices are saved alongside Tier 1)
print("Building Tier 2 splits...")

def build_joint_tier2_splits(per_splits: dict) -> dict:
    """Combine per-source Tier-2 splits by offsetting indices."""
    offsets = {"os1": 0, "os2": len(os1_nc), "fusar": len(os1_nc) + len(os2_nc)}
    def shift(src, split):
        return [i + offsets[src] for i in per_splits[src][split]]
    train_idx = shift("os1","train_idx") + shift("os2","train_idx") + shift("fusar","train_idx")
    val_idx   = shift("os1","val_idx")   + shift("os2","val_idx")   + shift("fusar","val_idx")
    test_idx  = shift("os1","test_idx")  + shift("os2","test_idx")  + shift("fusar","test_idx")
    jnc = pd.concat([os1_nc, os2_nc, fusar_nc], ignore_index=True)
    jnc["t2_label_id"] = jnc["label"].map(TIER2_LABEL_TO_INT)
    ds_train = SARDataset(jnc, "t2_label_id", transform=transform_train)
    ds_eval  = SARDataset(jnc, "t2_label_id", transform=transform_eval)
    return {
        "train_idx"    : train_idx,
        "val_idx"      : val_idx,
        "test_idx"     : test_idx,
        "train_loader" : DataLoader(Subset(ds_train, train_idx), batch_size=BATCH_SIZE,
                                    shuffle=True,  num_workers=_NW, pin_memory=_PIN,
                                    persistent_workers=bool(_NW)),
        "val_loader"   : DataLoader(Subset(ds_eval,  val_idx),   batch_size=BATCH_SIZE,
                                    shuffle=False, num_workers=_NW, pin_memory=_PIN,
                                    persistent_workers=bool(_NW)),
        "test_loader"  : DataLoader(Subset(ds_eval,  test_idx),  batch_size=BATCH_SIZE,
                                    shuffle=False, num_workers=_NW, pin_memory=_PIN,
                                    persistent_workers=bool(_NW)),
    }

t2_per = {
    "os1"  : build_splits(os1_nc,   "t2_label_id"),
    "os2"  : build_splits(os2_nc,   "t2_label_id"),
    "fusar": build_splits(fusar_nc, "t2_label_id"),
}
tier2_domains = {**t2_per, "joint": build_joint_tier2_splits(t2_per)}

# Print split sizes
for tier_label, domains in [("Tier 1", tier1_domains), ("Tier 2", tier2_domains)]:
    print(f"\n{tier_label}:")
    for k, v in domains.items():
        print(f"  {k}: train={len(v['train_idx'])}  val={len(v['val_idx'])}  test={len(v['test_idx'])}")

## Training Utilities

In [ ]:
class FocalLoss(nn.Module):
    def __init__(self, gamma: float = 2.0):
        super().__init__()
        self.gamma = gamma

    def forward(self, inputs: torch.Tensor, targets: torch.Tensor) -> torch.Tensor:
        ce = F.cross_entropy(inputs, targets, reduction="none")
        return ((1 - torch.exp(-ce)) ** self.gamma * ce).mean()


def free_cuda(model: Optional[nn.Module] = None) -> None:
    if model is not None:
        model.cpu()
    gc.collect()
    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats()


def train_model(
    model: nn.Module,
    run_name: str,
    class_names: List[str],
    train_loader: DataLoader,
    val_loader: DataLoader,
    weights_dir: Path,
    epochs: int = EPOCHS,
    lr: float = LR,
) -> Tuple[nn.Module, dict]:
    model     = model.to(DEVICE)
    criterion = FocalLoss(gamma=2.0)
    optimizer = Adam(
        filter(lambda p: p.requires_grad, model.parameters()),
        lr=lr, weight_decay=1e-4,
    )
    scheduler = CosineAnnealingLR(optimizer, T_max=epochs, eta_min=1e-6)
    best_val  = 0.0
    ckpt      = weights_dir / f"{run_name}_best.pth"
    history   = {"train_loss": [], "val_loss": [], "train_acc": [], "val_acc": []}

    for epoch in range(epochs):
        model.train()
        run_loss, correct, total = 0.0, 0, 0
        for inputs, labels in train_loader:
            inputs  = inputs.to(DEVICE, non_blocking=True)
            labels  = labels.to(DEVICE, non_blocking=True)
            optimizer.zero_grad()
            outputs = model(inputs)
            loss    = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            run_loss += loss.item()
            _, pred   = outputs.detach().max(1)
            total    += labels.size(0)
            correct  += pred.eq(labels).sum().item()
        scheduler.step()
        t_loss = run_loss / len(train_loader)
        t_acc  = 100.0 * correct / total

        model.eval()
        v_loss, correct, total = 0.0, 0, 0
        with torch.no_grad():
            for inputs, labels in val_loader:
                inputs  = inputs.to(DEVICE, non_blocking=True)
                labels  = labels.to(DEVICE, non_blocking=True)
                outputs = model(inputs)
                v_loss += criterion(outputs, labels).item()
                _, pred = outputs.max(1)
                total  += labels.size(0)
                correct += pred.eq(labels).sum().item()
        v_loss /= len(val_loader)
        v_acc   = 100.0 * correct / total

        if v_acc > best_val:
            best_val = v_acc
            torch.save(model.state_dict(), ckpt)

        history["train_loss"].append(t_loss)
        history["val_loss"].append(v_loss)
        history["train_acc"].append(t_acc)
        history["val_acc"].append(v_acc)

        print(
            f"  [{run_name}] {epoch+1:02d}/{epochs}  "
            f"loss {t_loss:.4f}/{v_loss:.4f}  "
            f"acc {t_acc:.1f}%/{v_acc:.1f}%  "
            f"lr {scheduler.get_last_lr()[0]:.2e}"
        )

    model.load_state_dict(torch.load(ckpt, map_location=DEVICE))
    return model, history


def save_curves(histories: dict, title: str, save_path: Path) -> None:
    colors = {"os1": "steelblue", "os2": "darkorange", "fusar": "forestgreen", "joint": "mediumpurple"}
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    fig.suptitle(title, fontsize=13)
    for key, h in histories.items():
        c = colors.get(key, "gray")
        ax1.plot(h["train_loss"], "--", color=c, alpha=0.6, label=f"{key} train")
        ax1.plot(h["val_loss"],         color=c,             label=f"{key} val")
        ax2.plot(h["train_acc"],  "--", color=c, alpha=0.6, label=f"{key} train")
        ax2.plot(h["val_acc"],          color=c,             label=f"{key} val")
    ax1.set_title("Loss");     ax1.set_xlabel("Epoch"); ax1.legend(fontsize=8)
    ax2.set_title("Accuracy"); ax2.set_xlabel("Epoch"); ax2.legend(fontsize=8)
    plt.tight_layout()
    plt.savefig(save_path, dpi=150)
    plt.show()

print("Utilities defined.")

## Tier 1 — ResNet-18

In [ ]:
resnet_t1_results = {}
resnet_t1_history = {}
resnet_dir = WEIGHTS_DIR / "resnet"
resnet_dir.mkdir(exist_ok=True)

print(f"{'='*55}")
print("  RESNET — Tier 1 (binary: cargo vs not-cargo)")
print(f"{'='*55}")

for domain_name, splits in tier1_domains.items():
    free_cuda()
    run   = f"resnet_tier1_{domain_name}"
    model = get_model("resnet", TIER1_NUM_CLASSES)
    model, history = train_model(model, run, TIER1_CLASS_NAMES,
                                 splits["train_loader"], splits["val_loader"], resnet_dir)
    resnet_t1_history[domain_name] = history
    free_cuda(model)

save_curves(resnet_t1_history, "ResNet — Tier 1", MISC_DIR / "resnet_tier1_curves.png")

## Tier 1 — ResNet-18 + CBAM (BAM)

In [ ]:
bam_t1_results = {}
bam_t1_history = {}
bam_dir = WEIGHTS_DIR / "bam"
bam_dir.mkdir(exist_ok=True)

print(f"{'='*55}")
print("  BAM — Tier 1 (binary: cargo vs not-cargo)")
print(f"{'='*55}")

for domain_name, splits in tier1_domains.items():
    free_cuda()
    run   = f"bam_tier1_{domain_name}"
    model = get_model("bam", TIER1_NUM_CLASSES)
    model, history = train_model(model, run, TIER1_CLASS_NAMES,
                                 splits["train_loader"], splits["val_loader"], bam_dir)
    bam_t1_history[domain_name] = history
    free_cuda(model)

save_curves(bam_t1_history, "BAM — Tier 1", MISC_DIR / "bam_tier1_curves.png")

## Tier 1 — Swin Transformer

In [ ]:
swin_t1_results = {}
swin_t1_history = {}
swin_dir = WEIGHTS_DIR / "swin"
swin_dir.mkdir(exist_ok=True)

print(f"{'='*55}")
print("  SWIN — Tier 1 (binary: cargo vs not-cargo)")
print(f"{'='*55}")

for domain_name, splits in tier1_domains.items():
    free_cuda()
    run   = f"swin_tier1_{domain_name}"
    model = get_model("swin", TIER1_NUM_CLASSES)
    model, history = train_model(model, run, TIER1_CLASS_NAMES,
                                 splits["train_loader"], splits["val_loader"], swin_dir)
    swin_t1_history[domain_name] = history
    free_cuda(model)

save_curves(swin_t1_history, "Swin — Tier 1", MISC_DIR / "swin_tier1_curves.png")

## Save Tier 1 History

In [ ]:
tier1_all_history = {
    "resnet": resnet_t1_history,
    "bam"   : bam_t1_history,
    "swin"  : swin_t1_history,
}

with open(MISC_DIR / "tier1_history.json", "w") as f:
    json.dump(tier1_all_history, f, indent=2)

# Save OUT_DIR path so Part 2 can find the weights
with open(Path(".").resolve() / "outputs" / "current_run.txt", "w") as f:
    f.write(str(OUT_DIR))

print(f"Tier 1 history saved.")
print(f"Run directory: {OUT_DIR}")
print("\nTier 1 best val accuracies:")
for arch, histories in tier1_all_history.items():
    for domain, h in histories.items():
        print(f"  {arch} {domain}: {max(h['val_acc']):.2f}%")